# Gradient Scaling Effects in Adaptive Spectral PINNs for Stiff Nonlinear ODEs

**Paper:** Yepes, I.M., Protopapas, P. (2026). *Gradient Scaling Effects in Adaptive Spectral PINNs for Stiff Nonlinear ODEs.* arXiv:2605.04502 [cs.LG] (ICLR 2026 AI&PDE Workshop).

**Carpeta origen:** `PINNs/4. Otros/Gradient Scaling Effects in Adaptive Spectral PINNs for Stiff.pdf`

## Como se usan las PINNs en este paper

El paper estudia un sistema **resorte-pendulo** rigido (stiff) en coordenadas polares (Eq. 1):

$$\ddot r = r\dot\theta^2-\frac{k}{m}(r-L_0)+g\cos\theta,\qquad \ddot\theta=-\frac{2\dot r\dot\theta}{r}-\frac{g}{r}\sin\theta$$

donde $k$ controla la rigidez del sistema. La condicion inicial se impone **de forma dura** mediante una funcion de compuerta (*IC gate*) $g(t)$ con $g(0)=0$:

$$\hat{\mathbf{u}}(t) = \mathbf{u}_0 + g(t)\,\tilde{\mathbf{u}}(t)$$

donde $\tilde{\mathbf{u}}(t)=[\tilde\rho(t),\tilde\theta(t)]$ es la salida cruda de la red. **El hallazgo central del paper** es que, aunque $g(t)$ no cambia el espacio de funciones admisibles (cualquier $g$ con $g(0)=0,g'(0)\neq0$ sirve para imponer la condicion inicial), **si cambia drasticamente la dinamica de optimizacion**: via el Neural Tangent Kernel (NTK), $g(t)$ reescala el Jacobiano del residuo en el tiempo fisico, alterando como el descenso de gradiente distribuye su enfasis a lo largo de la trayectoria. Comparan dos compuertas: **exponencial** $g(t)=1-e^{-t}$ (satura rapido, ponderacion casi uniforme) y **lineal** $g(t)=t$ (crece sin limite, enfatiza tiempos tardios), combinadas con una **red base (MLP)** y una **red espectral de Fourier** (features $\Phi(t)\in\mathbb{R}^D$ de frecuencias fijas o adaptativas, seguidas de una capa lineal).

El resultado experimental (Fig. 1): a rigidez moderada ($k=20$) la compuerta **exponencial** suele dar menor error, mientras que a rigidez alta ($k=60$) la compuerta **lineal** se vuelve preferible &mdash; una **reversion dependiente de la rigidez** en cual compuerta conviene, confirmada con tests de Wilcoxon pareados.

Este cuaderno reproduce fielmente: el sistema resorte-pendulo (Eq. 1), la compuerta de condicion inicial dura, la restriccion de positividad de $r$ via softplus, el modelo espectral de Fourier de frecuencias fijas, y compara **compuerta exponencial vs. lineal** en dos regimenes de rigidez ($k=20$ y $k=60$) para verificar la reversion reportada.

## Repositorio publico

El paper **incluye explicitamente** su repositorio de codigo:

- **isabelayepes/gradient-scaling-pinns** &mdash; https://github.com/isabelayepes/gradient-scaling-pinns

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib scipy

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Sistema resorte-pendulo (Eq. 1) y solucion de referencia de alta precision (DOP853)

In [ ]:
m, L0, g_grav, cr, ctheta = 1.0, 1.0, 9.81, 0.0, 0.0
T_max = 10.0
r0, theta0, rdot0, thetadot0 = 1.3, 0.6, 0.0, 0.0

def spring_pendulum_rhs(t, y, k):
    r, theta, rdot, thetadot = y
    rddot = r * thetadot**2 - (k / m) * (r - L0) + g_grav * np.cos(theta) - (cr / r) * rdot
    thetaddot = -2 * rdot * thetadot / r - (g_grav / r) * np.sin(theta) - ctheta * thetadot
    return [rdot, thetadot, rddot, thetaddot]

def reference_solution(k, n_eval=2000):
    t_eval = np.linspace(0, T_max, n_eval)
    sol = solve_ivp(spring_pendulum_rhs, [0, T_max], [r0, theta0, rdot0, thetadot0],
                     args=(k,), method='DOP853', t_eval=t_eval, rtol=1e-10, atol=1e-12)
    return t_eval, sol.y[0], sol.y[1]

## 2. Modelo espectral de Fourier con compuerta de condicion inicial dura (Seccion 2)

In [ ]:
r_min = 1e-4

def softplus_inv(y):
    return np.log(np.expm1(y))

rho0_latent = softplus_inv(r0 - r_min)  # ζ^{-1}(r0 - r_min), constante fija


class SpectralPINN(nn.Module):
    """Phi(t) con 32 frecuencias log-espaciadas (2 bandas), D=64 features (sin+cos), + cabeza lineal."""
    def __init__(self, gate='exp'):
        super().__init__()
        freqs = np.concatenate([np.geomspace(0.5, 5.0, 16), np.geomspace(5.0, 15.0, 16)])
        self.register_buffer('freqs', torch.tensor(freqs, dtype=torch.float32))
        self.head = nn.Linear(2 * len(freqs), 2)
        self.gate = gate

    def g_gate(self, t):
        return (1 - torch.exp(-t)) if self.gate == 'exp' else t

    def forward(self, t):
        phase = t * self.freqs                       # (N, D/2)
        Phi = torch.cat([torch.sin(phase), torch.cos(phase)], dim=1)
        u_tilde = self.head(Phi)                       # (N, 2): [rho_tilde, theta_tilde]
        gt = self.g_gate(t)
        rho_hat = rho0_latent + gt * u_tilde[:, 0:1]
        theta_hat = theta0 + gt * u_tilde[:, 1:2]
        r_hat = r_min + torch.nn.functional.softplus(rho_hat)   # Eq. de positividad, Seccion 2
        return r_hat, theta_hat


def d_dt(f, t):
    return torch.autograd.grad(f, t, grad_outputs=torch.ones_like(f),
                                create_graph=True, retain_graph=True)[0]

## 3. Perdida (Seccion 2.2): residuo fisico + penalizacion blanda de velocidad inicial

In [ ]:
def compute_loss(model, t_col, t_ic, k):
    r, theta = model(t_col)
    rdot = d_dt(r, t_col)
    thetadot = d_dt(theta, t_col)
    rddot = d_dt(rdot, t_col)
    thetaddot = d_dt(thetadot, t_col)

    res_r = rddot - (r * thetadot**2 - (k / m) * (r - L0) + g_grav * torch.cos(theta))
    res_theta = thetaddot - (-2 * rdot * thetadot / r - (g_grav / r) * torch.sin(theta))
    loss_phys = torch.mean(res_r**2) + torch.mean(res_theta**2)

    r_i, theta_i = model(t_ic)
    rdot_i = d_dt(r_i, t_ic)
    thetadot_i = d_dt(theta_i, t_ic)
    loss_ic_vel = (rdot_i - rdot0)**2 + (thetadot_i - thetadot0)**2

    return loss_phys + 50.0 * loss_ic_vel.squeeze()

## 4. Entrenamiento: compuerta exponencial vs. lineal, en dos regimenes de rigidez ($k=20$, $k=60$)

In [ ]:
def train(gate, k, epochs=4000, N_coll=800):
    model = SpectralPINN(gate=gate).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    t_ic = torch.zeros(1, 1, device=device, requires_grad=True)
    for epoch in range(epochs):
        opt.zero_grad()
        t_col = (torch.rand(N_coll, 1, device=device) * T_max).requires_grad_(True)
        loss = compute_loss(model, t_col, t_ic, k)
        loss.backward()
        opt.step()
        if epoch % 1000 == 0:
            print(f'[gate={gate}, k={k}] epoch {epoch:5d} | loss={loss.item():.4e}')
    return model


results = {}
for k in [20, 60]:
    for gate in ['exp', 'linear']:
        results[(k, gate)] = train(gate, k)

## 5. Resultados: error relativo L2 por compuerta y rigidez (cf. Fig. 1 del paper: reversion exp/lineal)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
errors = {}
for idx, k in enumerate([20, 60]):
    t_ref, r_ref, theta_ref = reference_solution(k)
    t_ref_t = torch.tensor(t_ref, dtype=torch.float32, device=device).view(-1, 1)
    ax = axes[idx]
    ax.plot(t_ref, r_ref, 'k-', label='referencia (DOP853)', linewidth=1.5)
    for gate, style in [('exp', '--'), ('linear', ':')]:
        model = results[(k, gate)]
        with torch.no_grad():
            r_pred, _ = model(t_ref_t)
        r_pred = r_pred.cpu().numpy().flatten()
        rel_l2 = 100 * np.linalg.norm(r_pred - r_ref) / np.linalg.norm(r_ref)
        errors[(k, gate)] = rel_l2
        ax.plot(t_ref, r_pred, style, label=f'{gate} (ReL2E={rel_l2:.2f}%)')
    ax.set_xlabel('t'); ax.set_ylabel('r(t)')
    ax.set_title(f'k={k}')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print('Errores relativos L2 en r(t):')
for (k, gate), err in errors.items():
    print(f'  k={k:3d}, gate={gate:7s}: {err:.2f}%')

**Nota honesta sobre los resultados:** en una prueba reducida (menos epocas que las 4000 configuradas arriba) la compuerta **exponencial** supero a la lineal en ambos regimenes de rigidez, sin mostrar todavia la **reversion** que reporta el paper a $k=60$. Esto es consistente con la propia mecanica del gate lineal $g(t)=t$: al crecer sin limite hasta $T=10$, la salida cruda de la red debe ser proporcionalmente mas pequena para representar la misma trayectoria fisica, lo que ralentiza la convergencia inicial y requiere mas iteraciones para alcanzar su ventaja a alta rigidez (el paper entrena 5000 actualizaciones de Adam). El **mecanismo central -- compuerta de condicion inicial dura con dos formas funcionales, restriccion de positividad via softplus, y modelo espectral de Fourier -- esta fielmente implementado**; confirmar cuantitativamente la reversion dependiente de la rigidez que reporta el paper requeriria el presupuesto de entrenamiento completo (5000 iteraciones) y, segun el paper, promediar sobre multiples semillas aleatorias.